# WAS-Mamba on BraTS -- Training on Kaggle

Kaggle-specific version of `train_brats.ipynb`. Two things are different from Colab here:

1. **No download needed** -- BraTS2021 is already a public Kaggle dataset. Attach it to this notebook instead of downloading:
   Right sidebar -> **Add Input** -> search **"BRaTS 2021 Task 1"** -> add `dschettler8845/brats-2021-task1`.
   It mounts read-only at `/kaggle/input/datasets/dschettler8845/brats-2021-task1/` (confirmed via `!find /kaggle/input -maxdepth 4` -- Kaggle nests it under `datasets/<owner>/<slug>/`, not directly under `/kaggle/input/<slug>/`). Its files are `.tar` archives, not loose `.nii.gz` -- section 4 below extracts them.
2. **Checkpoints** go to `/kaggle/working/` -- this persists for the life of the interactive session, but is only kept permanently if you **Save Version** (top right) before the session ends. Kaggle GPU sessions also have a **weekly quota** (not just a per-session limit like Colab) -- check Settings for how much you have left.

**Before running:**
- Notebook Settings (right sidebar) -> **Accelerator** -> GPU T4 x2 (or P100)
- Notebook Settings -> **Internet** -> **On** (needed for `git clone` and `pip install`)
- Add the BraTS2021 dataset as described above

## 1. Get the code

In [ ]:
import os

REPO_URL = "github.com/sachinn854/Brain-Tumor-Segmentatiton.git"
REPO_DIR = "/kaggle/working/Brain-Tumor-Segmentatiton"

if not os.path.isdir(REPO_DIR):
    get_ipython().system(f'git clone https://{REPO_URL} {REPO_DIR}')
else:
    get_ipython().system(f'git -C {REPO_DIR} pull')

get_ipython().run_line_magic('cd', REPO_DIR)

## 2. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install dependencies

Same reasoning as the Colab notebooks: `--no-build-isolation` so the build sees Kaggle's pre-installed torch, and `TORCH_CUDA_ARCH_LIST` pinned to the actual GPU so `mamba-ssm`/`causal-conv1d` don't compile for ~10 architectures they'll never use.

In [ ]:
import torch

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"Building only for compute capability {major}.{minor}")

!pip install -q einops timm nibabel ninja packaging
!pip install -q causal-conv1d --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

## 4. Extract a couple of small test cases and reorganize

`/kaggle/input/` is read-only and its files are `.tar` archives. For the smoke test we only need a couple of cases, so this extracts just the two small per-case `.tar` files (~10MB each: `BraTS2021_00495.tar`, `BraTS2021_00621.tar`) -- **not** `BraTS2021_Training_Data.tar` (13.4GB, all 1251 cases), which is skipped here on purpose. That one gets extracted separately in section 6 when you're ready for the real run.

Extracted into `/kaggle/working/BraTS2021/<case_id>/<files>`, matching what `BratsDataset` expects.

In [ ]:
import glob
import re
import shutil
import tarfile

KAGGLE_INPUT = '/kaggle/input/datasets/dschettler8845/brats-2021-task1'
DATA_ROOT = '/kaggle/working/BraTS2021'
RAW_EXTRACT_DIR = '/kaggle/working/brats_raw_extract'

if not os.path.isdir(KAGGLE_INPUT):
    raise SystemExit(
        f"{KAGGLE_INPUT} not found -- add the dataset first: "
        "right sidebar -> Add Input -> search 'BRaTS 2021 Task 1' -> "
        "add dschettler8845/brats-2021-task1"
    )

if not os.path.isdir(DATA_ROOT):
    os.makedirs(DATA_ROOT, exist_ok=True)
    os.makedirs(RAW_EXTRACT_DIR, exist_ok=True)

    SIZE_LIMIT_BYTES = 1 * 1024 ** 3  # skip anything over 1GB -- i.e. skip the 13.4GB full archive here
    tar_paths = glob.glob(f'{KAGGLE_INPUT}/**/*.tar', recursive=True)
    small_tars = [p for p in tar_paths if os.path.getsize(p) < SIZE_LIMIT_BYTES]
    print(f'Found {len(tar_paths)} .tar files, extracting {len(small_tars)} small one(s) for the smoke test:')
    for p in small_tars:
        print(f'  {p} ({os.path.getsize(p) / 1e6:.1f} MB)')
        with tarfile.open(p) as tf:
            tf.extractall(RAW_EXTRACT_DIR)

    all_files = glob.glob(f'{RAW_EXTRACT_DIR}/**/*.nii.gz', recursive=True)
    print(f'Found {len(all_files)} .nii.gz files after extraction')

    copied, skipped = 0, 0
    for fpath in all_files:
        fname = os.path.basename(fpath)
        match = re.match(r'(BraTS2021_\d+)_', fname)
        if not match:
            skipped += 1
            continue
        case_id = match.group(1)
        case_dir = os.path.join(DATA_ROOT, case_id)
        os.makedirs(case_dir, exist_ok=True)
        dest = os.path.join(case_dir, fname)
        if not os.path.exists(dest):
            shutil.move(fpath, dest)
            copied += 1

    print(f'Moved {copied} files, skipped {skipped} (unrecognized naming)')
    shutil.rmtree(RAW_EXTRACT_DIR, ignore_errors=True)
else:
    print(f'{DATA_ROOT} already exists, skipping.')

n_cases = len([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f'{n_cases} cases ready in {DATA_ROOT}')

Note: with only 2 cases extracted, the 80:5:15 split gives 1 train / 0 val / 1 test -- val will be empty, so validation Dice will just print as 0.0 for every class. That's expected and fine here; this step is only checking that the training loop itself runs without crashing, not producing meaningful numbers. Real numbers need the full dataset (section 6).

## 5. Smoke-test the training loop (5 epochs)

In [ ]:
CHECKPOINT_DIR = '/kaggle/working/wasmamba_checkpoints'

!python -m src.engine.train \
    --data_path {DATA_ROOT} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --epochs 5

If that ran cleanly with no OOM and printed per-epoch train/val loss and per-class Dice: the loop works on Kaggle's GPU too. Remove the smoke-test checkpoint before a real run, same reason as on Colab -- otherwise the next run "resumes" from epoch 5 instead of starting fresh.

In [ ]:
smoke_test_ckpt = os.path.join(CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(smoke_test_ckpt):
    os.remove(smoke_test_ckpt)
    print('Removed smoke-test checkpoint. Ready for a real run.')
else:
    print('No checkpoint found -- nothing to remove.')

## 6. Real training -- extract the full dataset first

Section 4 only extracted 2 cases for the smoke test. The real run needs all 1251, from `BraTS2021_Training_Data.tar` (13.4GB) -- skipped there on purpose, extracted here instead. This will take a while and use significant disk in `/kaggle/working/` -- check your remaining disk quota if this fails partway.

In [ ]:
FULL_DATA_ROOT = '/kaggle/working/BraTS2021_full'
FULL_TAR = os.path.join(KAGGLE_INPUT, 'BraTS2021_Training_Data.tar')

if not os.path.isdir(FULL_DATA_ROOT):
    os.makedirs(FULL_DATA_ROOT, exist_ok=True)
    os.makedirs(RAW_EXTRACT_DIR, exist_ok=True)

    print(f'Extracting {FULL_TAR} (13.4GB, this takes a while)...')
    with tarfile.open(FULL_TAR) as tf:
        tf.extractall(RAW_EXTRACT_DIR)

    all_files = glob.glob(f'{RAW_EXTRACT_DIR}/**/*.nii.gz', recursive=True)
    print(f'Found {len(all_files)} .nii.gz files, organizing into per-case folders...')

    moved, skipped = 0, 0
    for fpath in all_files:
        fname = os.path.basename(fpath)
        match = re.match(r'(BraTS2021_\d+)_', fname)
        if not match:
            skipped += 1
            continue
        case_id = match.group(1)
        case_dir = os.path.join(FULL_DATA_ROOT, case_id)
        os.makedirs(case_dir, exist_ok=True)
        dest = os.path.join(case_dir, fname)
        if not os.path.exists(dest):
            shutil.move(fpath, dest)
            moved += 1

    print(f'Moved {moved} files, skipped {skipped}')
    shutil.rmtree(RAW_EXTRACT_DIR, ignore_errors=True)
else:
    print(f'{FULL_DATA_ROOT} already exists, skipping extraction.')

n_cases = len([d for d in os.listdir(FULL_DATA_ROOT) if os.path.isdir(os.path.join(FULL_DATA_ROOT, d))])
print(f'{n_cases} cases ready in {FULL_DATA_ROOT}')

In [ ]:
!python -m src.engine.train \
    --data_path {FULL_DATA_ROOT} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --epochs 5